### Coleta de dados de  Precipitação

Será utilizado o dataset derived-era5-single-levels-daily-statistics do ERA5

Documentação em:
https://cds.climate.copernicus.eu/datasets/derived-era5-single-levels-daily-statistics?tab=documentation

Os dados extraídos:
<pre>
- Umidade   -> variável temperature,
               O processo de conversão para percentual será detalhado no passo que executa a conversão
</pre>

Os dados serão coletados por Ano e Mês 

Os dados requisitados estão no retangulo geográfico geográfico [6, -74, -34, -35] -> [Norte, Oeste, Sul, Leste] em graus onde está o Brasil


In [ ]:
import cdsapi
import os, sys
import xarray as xr
from datetime import datetime, timedelta
from pyspark.sql import functions as F

In [ ]:
# Cria a conexão Spark

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("Precipitacao")

In [ ]:
# Os arquivos utilizados durante o processamento serão removidos no final do notebook
remover_arquivos = []

Requisição dos dados da variável derived-era5-single-levels-daily-statistics do ERA5 utilizando a biblioteca cdsapi

In [ ]:
PROJECT_PATH   = os.getcwd()
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

print(PROJECT_PATH)
print(DATA_PATH_ROOT)

In [ ]:
def obter_mes_dia(ano: int):
    hoje = datetime.now()

    # Se dia 01 ou 02, considerar mês anterior
    if hoje.day in (1, 2):
        data_referencia = hoje.replace(day=1) - timedelta(days=1)
    else:
        data_referencia = hoje

    ano_ref = data_referencia.year
    mes_ref = data_referencia.month

    # Lista de meses
    if ano < ano_ref:
        meses = [f"{m:02d}" for m in range(1, 13)]
        dias = [f"{d:02d}" for d in range(1, 32)]

    elif ano == ano_ref:
        meses = [f"{m:02d}" for m in range(1, mes_ref + 1)]

        # Último dia válido considerando D-2
        data_limite = hoje - timedelta(days=2)

        # Se estiver no mês da data limite, retorna apenas até D-2
        dias = [f"{d:02d}" for d in range(1, data_limite.day + 1)]

    else:
        raise ValueError(f"Ano futuro não permitido: {ano}")

    return meses, dias

def get_EAC4(year, client):

    month_list, day_list = obter_mes_dia(year)

    dataset = "reanalysis-era5-single-levels"
    request = {
        "product_type": "reanalysis",
        "variable": [
            "total_precipitation"
        ],
        "year":  f"{year}",
        "month": month_list,
        "day":   day_list,
        "time": ["10:00"],
        "data_format": "netcdf",
        "download_format": "unarchived",

        # Retangulo geográfico definido por Norte, Oeste, Sul e Leste em graus onde está o Brasil
        "area": [6      # Norte
                ,-74    # Oeste
                ,-34    # Sul
                ,-38]   # Leste
    }

    ret_download = client.retrieve(dataset, request).download()
    return ret_download

def convert_netcdf4_Spark(file_name):
    with xr.open_dataset(file_name
                        ,engine="netcdf4"
                        # ,chunks={"time": 365
                        #         ,"latitude": 100
                        #         ,"longitude": 100 }
                        ) as ds:
        
        # Transforma o Dataset em um Spark Dataframe
        df_dask   = ds.to_dask_dataframe()
        df_dask_c = df_dask.compute()
        df_spark  = spark.createDataFrame(df_dask_c)    
    return df_spark    

def convert_unit(df_precipitacao):
    drop_cols = ["valid_time", "number", "tp"]
    df_precipitacao_mm = \
        (df_precipitacao
            .withColumns({"indicador": F.lit("precipitacao")
                        ,"valor": (F.col("tp") * F.lit(1000)).cast('double')
                        ,"unidade_medida": F.lit("mm")
                        ,"data_medicao": F.col("valid_time").cast("date")}
                        )
            .drop(*drop_cols)
        )

    return df_precipitacao_mm

def write_data_csv(df_precipitacao_mm, write_path, file_name):
    df_precipitacao_mm.toPandas().to_csv(f"{write_path}\{file_name}")

In [11]:
client = cdsapi.Client(
    url = os.getenv("ECMWF_DATASTORES_URL"),
    key = os.getenv("ECMWF_DATASTORES_KEY"),
)

for year in range(1991, 2014): # Ajuste o intervalo de anos conforme necessário
    # for month in range(1,13):
    start = datetime.now()
    print("\nStart download - year: ", year, start)    

    ret_download = get_EAC4(year, client)
    nc_file_name = r"{DATA_PATH_ROOT}ERA5-precipitacao\arquivos_nc\ERA5_precipitacao_{year}.nc".format(DATA_PATH_ROOT = DATA_PATH_ROOT, year = year)
    os.rename(ret_download, nc_file_name)

    df_precipitacao = convert_netcdf4_Spark(nc_file_name)

    df_precipitacao_celsius = convert_unit(df_precipitacao)

    csv_path      = r"{DATA_PATH_ROOT}\ERA5-precipitacao\arquivos_csv".format(DATA_PATH_ROOT = DATA_PATH_ROOT)
    csv_file_name = f"ERA5_precipitacao_{year}.csv"

    write_data_csv(df_precipitacao_celsius, csv_path, csv_file_name)

    final = datetime.now()
    print("End download and transformations : ", final, " - Duration: ", final-start, "\n")

    # time.sleep(300)


Start download - year:  1991 2026-08-07 15:53:24.293343


2026-08-07 15:53:25,129 INFO Request ID is 0d2dd8ed-9e2e-444e-b3bf-68877d2cbc3e
2026-08-07 15:53:25,459 INFO status has been updated to accepted
2026-08-07 15:53:48,296 INFO status has been updated to running
2026-08-07 15:55:21,758 INFO status has been updated to successful


End download and transformations :  2026-08-07 15:56:23.115339  - Duration:  0:02:58.821996 


Start download - year:  1992 2026-08-07 15:56:23.131362


2026-08-07 15:56:24,657 INFO Request ID is b6732209-b94a-449f-8908-fcd220664fec
2026-08-07 15:56:24,847 INFO status has been updated to accepted
2026-08-07 15:56:39,136 INFO status has been updated to running
2026-08-07 15:57:41,947 INFO status has been updated to successful


End download and transformations :  2026-08-07 15:58:38.500566  - Duration:  0:02:15.369204 


Start download - year:  1993 2026-08-07 15:58:38.510004


...

End download and transformations :  2026-08-07 16:44:06.387156  - Duration:  0:02:05.857225 


Start download - year:  2012 2026-08-07 16:44:06.405910


2026-08-07 16:44:08,922 INFO Request ID is 14ad74f4-8196-4566-8d90-989ea3619b79
2026-08-07 16:44:09,102 INFO status has been updated to accepted
2026-08-07 16:44:23,471 INFO status has been updated to running
2026-08-07 16:45:00,749 INFO status has been updated to successful


End download and transformations :  2026-08-07 16:45:47.464836  - Duration:  0:01:41.058926 


Start download - year:  2013 2026-08-07 16:45:47.464836


2026-08-07 16:45:49,030 INFO Request ID is 3ebcaacf-3a87-4318-a90b-af60a7fd5b82
2026-08-07 16:45:49,229 INFO status has been updated to accepted
2026-08-07 16:46:14,671 INFO status has been updated to running
2026-08-07 16:47:09,371 INFO status has been updated to successful


End download and transformations :  2026-08-07 16:48:14.779266  - Duration:  0:02:27.314430 

